# AlphaTrain Pillar 2j: High-Contrast Value Head

Fix the "Mid-Game Blob" where 84% of training values were in [190-240].

**Key changes from 2i:**
1. **γ=0.95** (was 0.99) — half-life 14 turns, spreads value distribution
2. **Categorical head (64 bins)** (was scalar sigmoid) — better for MCTS backup
3. **max_score=100** (was 2000) — 1.59 pts/bin resolution (was 31.7)
4. **val_weight=0.01** (was 0.001) — 10x stronger value gradient
5. **30% endgame oversampling** — death spiral positions 6x overrepresented

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code (UPDATED with new dataset/train changes)
2. `expert_v2_pairwise_g095.pt.gz` — γ=0.95 data (880 MB compressed)
3. `pillar2i_best.pt` — base model (already on Drive from 2i training)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model (always fresh)
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['pillar2i_best.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    print(f'Copying {f}...')
    shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

# Decompress γ=0.95 data (always fresh)
gz_src = os.path.join(DRIVE, 'expert_v2_pairwise_g095.pt.gz')
pt_dst = '/content/alphatrain/data/expert_v2_pairwise_g095.pt'
print('Decompressing expert_v2_pairwise_g095.pt.gz...')
t0 = time.time()
!gzip -dc {gz_src} > {pt_dst}
print(f'Done in {time.time()-t0:.0f}s: {os.path.getsize(pt_dst)/1e9:.1f} GB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

# Verify data
d = torch.load('/content/alphatrain/data/expert_v2_pairwise_g095.pt', weights_only=True, map_location='cpu')
print(f"\nExpert V2 (γ=0.95): {d['boards'].shape[0]:,} states, {d['n_pairs']:,} pairs")
print(f"  max_score={d['max_score']}, bins={d['num_value_bins']}, gamma={d['gamma']}")
print(f"  has turns_remaining: {'turns_remaining' in d}")
if 'turns_remaining' in d:
    tr = d['turns_remaining']
    print(f"  endgame (≤100 turns): {(tr <= 100).sum().item():,} ({100*(tr <= 100).float().mean():.1f}%)")
del d

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2j: High-Contrast Value Head
# γ=0.95 (half-life 14 turns), categorical 64-bin head, max_score=100
# 30% endgame oversampling, val_weight=0.01
# Warm start from Pillar 2i (keeps improved policy, reinits value_fc2)
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/expert_v2_pairwise_g095.pt \
    --gpu-data --amp --compile \
    --value-bins 64 --val-weight 0.01 --rank-weight 1.0 \
    --endgame-fraction 0.3 --endgame-threshold 100 \
    --resume alphatrain/data/pillar2i_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2j_best.pt \
    --save-dir /content/checkpoints/pillar2j

In [ ]:
# Evaluate policy score
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/pillar2j/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2j/{f}'
    dst = f'{DRIVE}/pillar2j_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')